# RAG with OpenAI — Native Implementation

**RAG = Retrieval-Augmented Generation**

Instead of asking the AI from memory, we:
1. Store our own documents as *embeddings* (numbers that capture meaning)
2. When a question comes in, find the most relevant chunks
3. Hand those chunks to the LLM so it can answer with real facts

```
Your Documents → Chunks → Embeddings → Vector Store
                                              ↓
User Question → Embed → Search → Top Chunks → GPT → Answer
```

---
## Step 1 — Install & Import
Install the packages we need, then load them.

In [ ]:
# Install required libraries (run once)
# %pip install openai numpy python-dotenv tiktoken --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import math
import textwrap
from typing import List, Dict, Tuple
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

# Load API key from .env file  ← create a .env file with: OPENAI_API_KEY=sk-...
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ Setup complete. OpenAI client ready.")

✓ Setup complete. OpenAI client ready.


---
## Step 2 — Load Documents

We define the knowledge base here as plain text strings.  
In a real project you'd load `.txt` / `.pdf` files from disk.  
Each entry has a **source** label so we can cite it later.

In [2]:
# ── Sample knowledge base ────────────────────────────────────────────────
# Each document is a dict with "source" (citation label) and "text" (content).
# Replace or extend these with your own documents.

DOCUMENTS = [
    {
        "source": "AI_Basics.txt",
        "text": (
            "Artificial Intelligence (AI) is the simulation of human intelligence by machines. "
            "Machine Learning (ML) is a subset of AI where models learn from data. "
            "Deep Learning is a subset of ML that uses neural networks with many layers. "
            "Neural networks are inspired by the structure of the human brain, consisting of "
            "interconnected nodes called neurons that process information."
        ),
    },
    {
        "source": "NLP_Overview.txt",
        "text": (
            "Natural Language Processing (NLP) enables computers to understand human language. "
            "Transformers, introduced in the paper 'Attention Is All You Need' (2017), revolutionized NLP. "
            "BERT (Bidirectional Encoder Representations from Transformers) was released by Google in 2018. "
            "GPT (Generative Pre-trained Transformer) models are developed by OpenAI and excel at text generation. "
            "Tokenization is the process of splitting text into smaller units called tokens before feeding them to a model."
        ),
    },
    {
        "source": "RAG_Concepts.txt",
        "text": (
            "Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model. "
            "Embeddings are numerical vector representations of text that capture semantic meaning. "
            "Cosine similarity measures the angle between two vectors — a score near 1 means very similar. "
            "Chunking is the process of splitting long documents into smaller overlapping pieces for better retrieval. "
            "Vector databases like Pinecone, Weaviate, and FAISS store and search embeddings efficiently."
        ),
    },
    {
        "source": "OpenAI_Models.txt",
        "text": (
            "OpenAI offers several embedding models including text-embedding-3-small and text-embedding-3-large. "
            "text-embedding-3-small produces 1536-dimensional vectors and is cost-efficient. "
            "text-embedding-3-large produces 3072-dimensional vectors with higher accuracy. "
            "GPT-4o is OpenAI's flagship multimodal model capable of processing text and images. "
            "The Chat Completions API accepts a list of messages and returns a model-generated response."
        ),
    },
    {
        "source": "Production_Tips.txt",
        "text": (
            "In production RAG systems, chunk size is critical — too small loses context, too large loses precision. "
            "Typical chunk sizes range from 200 to 500 tokens with a 50-token overlap. "
            "Re-ranking retrieved chunks with a cross-encoder model improves answer quality. "
            "Hybrid search combines dense (embedding) retrieval with sparse (keyword) retrieval for better coverage. "
            "Monitoring hallucinations and retrieval quality with evals is essential before going to production."
        ),
    },
]

print(f"✓ Loaded {len(DOCUMENTS)} documents.")
for doc in DOCUMENTS:
    print(f"  📄 {doc['source']} — {len(doc['text'].split())} words")

✓ Loaded 5 documents.
  📄 AI_Basics.txt — 58 words
  📄 NLP_Overview.txt — 67 words
  📄 RAG_Concepts.txt — 65 words
  📄 OpenAI_Models.txt — 49 words
  📄 Production_Tips.txt — 66 words


---
## Step 3 — Chunk the Documents

**Why chunk?** Models have context limits and retrieval works better on focused passages.  
We split each document into overlapping windows measured in *tokens* (not characters),  
because that's how OpenAI's embedding model counts length.

In [3]:
def chunk_text(
    text: str,
    source: str,
    chunk_size: int = 100,   # tokens per chunk
    overlap: int = 20,       # tokens shared between consecutive chunks
    model: str = "text-embedding-3-small",
) -> List[Dict]:
    """
    Split text into overlapping token-based chunks.

    Steps:
    1. Tokenize the full text with tiktoken.
    2. Slide a window of `chunk_size` tokens, stepping by (chunk_size - overlap).
    3. Decode each window back to a string.
    4. Return a list of chunk dicts that also carry the source label and chunk index.
    """
    enc = tiktoken.encoding_for_model(model)
    tokens = enc.encode(text)

    chunks = []
    step = chunk_size - overlap          # how far we advance each iteration
    chunk_idx = 0

    for start in range(0, len(tokens), step):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        if not chunk_tokens:
            break

        chunk_text_str = enc.decode(chunk_tokens)
        chunks.append({
            "source": source,
            "chunk_id": chunk_idx,
            "text": chunk_text_str,
            "token_count": len(chunk_tokens),
        })
        chunk_idx += 1

        if end >= len(tokens):           # we've covered the whole document
            break

    return chunks


# ── Apply chunking to all documents ─────────────────────────────────────
all_chunks: List[Dict] = []

for doc in DOCUMENTS:
    doc_chunks = chunk_text(doc["text"], doc["source"])
    all_chunks.extend(doc_chunks)

print(f"✓ Created {len(all_chunks)} chunks from {len(DOCUMENTS)} documents.")
print("\nFirst chunk preview:")
print(json.dumps(all_chunks[0], indent=2))

✓ Created 5 chunks from 5 documents.

First chunk preview:
{
  "source": "AI_Basics.txt",
  "chunk_id": 0,
  "text": "Artificial Intelligence (AI) is the simulation of human intelligence by machines. Machine Learning (ML) is a subset of AI where models learn from data. Deep Learning is a subset of ML that uses neural networks with many layers. Neural networks are inspired by the structure of the human brain, consisting of interconnected nodes called neurons that process information.",
  "token_count": 68
}


---
## Step 4 — Generate Embeddings

**What is an embedding?** A list of ~1500 numbers that encodes the *meaning* of a text.  
Similar texts → similar vectors.  

We call OpenAI's embedding API in **batches** to reduce the number of API calls.

In [4]:
EMBEDDING_MODEL = "text-embedding-3-small"   # 1536-dim, fast and cheap


def get_embeddings_batch(
    texts: List[str],
    model: str = EMBEDDING_MODEL,
    batch_size: int = 100,
) -> List[List[float]]:
    """
    Send texts to OpenAI in batches and return their embedding vectors.

    Why batches? Sending all texts in one shot hits API payload limits.
    Batching also makes it easy to add retry logic per batch.
    """
    all_embeddings: List[List[float]] = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        # The API returns items sorted by index, so order is preserved.
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks…")

    return all_embeddings


# ── Embed all chunks ─────────────────────────────────────────────────────
print("Generating embeddings (this makes one API call)…")
chunk_texts = [c["text"] for c in all_chunks]
embeddings = get_embeddings_batch(chunk_texts)

# Attach the embedding vector back to each chunk dict
for chunk, emb in zip(all_chunks, embeddings):
    chunk["embedding"] = emb

print(f"\n✓ Done! Each embedding has {len(embeddings[0])} dimensions.")

Generating embeddings (this makes one API call)…


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

---
## Step 5 — Build an In-Memory Vector Store

A **vector store** is just a place to keep our chunks + embeddings so we can search them.  
Here we use plain Python (no external DB needed).  
Production systems would use Pinecone, Weaviate, or FAISS for millions of vectors.

In [ ]:
class SimpleVectorStore:
    """
    Minimal in-memory vector store.

    Stores: a list of chunks, each already containing its embedding vector.
    Searches: computes cosine similarity between a query vector and every stored vector,
              then returns the top-k most similar chunks.

    Cosine similarity formula:
        sim(A, B) = dot(A, B) / (|A| * |B|)
    Result is between -1 and 1; closer to 1 = more similar.
    """

    def __init__(self):
        self.chunks: List[Dict] = []

    def add(self, chunks: List[Dict]):
        """Add a list of chunk dicts (must already have an 'embedding' key)."""
        self.chunks.extend(chunks)
        print(f"✓ Vector store now holds {len(self.chunks)} chunks.")

    @staticmethod
    def _cosine_similarity(vec_a: List[float], vec_b: List[float]) -> float:
        """Pure-Python cosine similarity (no numpy needed)."""
        dot  = sum(a * b for a, b in zip(vec_a, vec_b))
        mag_a = math.sqrt(sum(a * a for a in vec_a))
        mag_b = math.sqrt(sum(b * b for b in vec_b))
        if mag_a == 0 or mag_b == 0:
            return 0.0
        return dot / (mag_a * mag_b)

    def search(self, query_embedding: List[float], top_k: int = 3) -> List[Dict]:
        """
        Return the top_k most similar chunks for the given query embedding.
        Each returned dict gets a 'score' key with the similarity value.
        """
        scored = [
            {**chunk, "score": self._cosine_similarity(query_embedding, chunk["embedding"])}
            for chunk in self.chunks
        ]
        # Sort descending by similarity score, return top k
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored[:top_k]


# ── Populate the store ───────────────────────────────────────────────────
vector_store = SimpleVectorStore()
vector_store.add(all_chunks)

---
## Step 6 — Retrieval Function

Given a question, embed it with the same model, then search the store.

In [ ]:
def retrieve(
    query: str,
    store: SimpleVectorStore,
    top_k: int = 3,
    model: str = EMBEDDING_MODEL,
) -> List[Dict]:
    """
    1. Embed the user's question.
    2. Find the top_k most similar chunks in the vector store.
    3. Return those chunks (with source + similarity score).

    Important: use the SAME embedding model for both documents and queries.
    """
    response = client.embeddings.create(model=model, input=[query])
    query_embedding = response.data[0].embedding
    results = store.search(query_embedding, top_k=top_k)
    return results


# ── Quick retrieval test ─────────────────────────────────────────────────
test_query = "What are embeddings?"
test_results = retrieve(test_query, vector_store)

print(f'Top {len(test_results)} chunks for: "{test_query}"\n')
for i, chunk in enumerate(test_results, 1):
    print(f"[{i}] Score: {chunk['score']:.4f} | Source: {chunk['source']}")
    print(f"     {chunk['text'][:120]}…\n")

---
## Step 7 — RAG Answer Function

Now we combine retrieval with generation:
1. Retrieve relevant chunks
2. Build a prompt that includes those chunks as *context*
3. Ask GPT to answer — using only the provided context

In [ ]:
CHAT_MODEL = "gpt-4o-mini"   # cheap and fast; swap for gpt-4o for higher quality


def format_context(chunks: List[Dict]) -> str:
    """
    Turn retrieved chunks into a readable context block for the prompt.
    Each chunk is labelled with its source file so GPT can cite it.
    """
    parts = []
    for i, chunk in enumerate(chunks, 1):
        parts.append(f"[Source {i}: {chunk['source']}]\n{chunk['text']}")
    return "\n\n".join(parts)


def rag_answer(
    question: str,
    store: SimpleVectorStore,
    top_k: int = 3,
    chat_model: str = CHAT_MODEL,
) -> Dict:
    """
    Full RAG pipeline:
      retrieve → format context → call Chat Completions → return answer + sources.

    Returns a dict with:
      - 'answer':  the model's response string
      - 'sources': list of (source_file, similarity_score) tuples
      - 'chunks':  the raw retrieved chunks (for inspection)
    """
    # ── Step A: Retrieve relevant chunks ────────────────────────────────
    chunks = retrieve(question, store, top_k=top_k)
    context = format_context(chunks)

    # ── Step B: Build the prompt ─────────────────────────────────────────
    system_prompt = (
        "You are a helpful assistant. Answer the user's question ONLY using the "
        "provided context. If the context doesn't contain the answer, say so clearly. "
        "Always cite the source number (e.g. [Source 1]) for every fact you state."
    )

    user_message = (
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    # ── Step C: Call the Chat Completions API ─────────────────────────────
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,   # 0 = deterministic / factual; increase for creativity
    )

    answer = response.choices[0].message.content

    # ── Step D: Collect source citations ─────────────────────────────────
    sources = [(c["source"], round(c["score"], 4)) for c in chunks]

    return {"answer": answer, "sources": sources, "chunks": chunks}


print("✓ RAG function defined. Proceed to the next cell to run example queries.")

---
## Step 8 — Run Example Queries

Let's ask five different questions and see the grounded answers with citations.

In [ ]:
def pretty_print(result: Dict, question: str):
    """Helper to display a RAG result in a readable format."""
    sep = "=" * 70
    print(sep)
    print(f"❓ QUESTION: {question}")
    print(sep)
    print(f"\n💬 ANSWER:\n{result['answer']}")
    print("\n📚 SOURCES USED:")
    for src, score in result["sources"]:
        print(f"   • {src}  (similarity: {score})")
    print()


EXAMPLE_QUESTIONS = [
    "What is the difference between AI, ML, and Deep Learning?",
    "What embedding models does OpenAI offer and how many dimensions do they have?",
    "What is cosine similarity and why is it used in RAG?",
    "What is a good chunk size for production RAG systems?",
    "Who introduced the Transformer architecture and when?",
]

results = []
for q in EXAMPLE_QUESTIONS:
    res = rag_answer(q, vector_store)
    results.append({"question": q, **res})
    pretty_print(res, q)

---
## Step 9 — Interactive Q&A Loop

Ask your own questions! Type `quit` to stop.

In [ ]:
print("Interactive RAG — type your question and press Enter. Type 'quit' to stop.\n")

while True:
    user_q = input("Your question: ").strip()
    if not user_q or user_q.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    res = rag_answer(user_q, vector_store)
    pretty_print(res, user_q)

---
## Step 10 — Save Results to JSON

Export all Q&A pairs (without the large embedding vectors) for the submission.

In [ ]:
# Prepare a clean export — strip embedding vectors (too large to store)
export = []
for r in results:
    export.append({
        "question": r["question"],
        "answer":   r["answer"],
        "sources":  r["sources"],
        "top_chunks": [
            {"source": c["source"], "chunk_id": c["chunk_id"],
             "score": round(c["score"], 4), "text": c["text"]}
            for c in r["chunks"]
        ],
    })

output_path = "rag_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to {output_path}")
print(f"  {len(export)} Q&A pairs exported.")

---
## Summary — What we built

| Step | What happened | Key API / tool |
|------|--------------|----------------|
| 1 | Set up environment | `openai`, `tiktoken` |
| 2 | Loaded 5 text documents | plain Python |
| 3 | Chunked docs into 100-token windows with 20-token overlap | `tiktoken` |
| 4 | Turned every chunk into a 1536-dim embedding vector | `text-embedding-3-small` |
| 5 | Stored chunks + vectors in a simple in-memory store | pure Python |
| 6 | Embedded the query and retrieved top-3 similar chunks | cosine similarity |
| 7 | Fed context + question into GPT to get a cited answer | `gpt-4o-mini` |
| 8 | Ran 5 example questions | — |
| 9 | Interactive Q&A loop | — |
| 10 | Exported results to JSON | — |

**Next steps for production:**
- Replace in-memory store with FAISS or Pinecone
- Load real PDFs with `pypdf2`
- Add re-ranking for better precision
- Add streaming for faster UI responses